# Step 7: why does fine-tuning break threshold reuse, and what fixes it?

**Question.** After a calibrated model is adapted, when does the old conformal threshold stop giving 90% coverage, and is the cause BatchNorm, re-training on data the model already saw, or neither?

**Data roles (no overlap, per seed).** The MedMNIST training set is split, stratified by class, into **A** (50%, trains the base model and is the *seen* data) and **B** (50%, *fresh* data, never seen by the base model). B is halved into **B1** (fresh fine-tuning data for the splitting baseline) and **B2** (labeled calibration data for the splitting baseline). The pooled validation + test images are the calibration/evaluation pool, split 50/50 at random 40 times. MedMNIST has no patient identifiers (DermaMNIST has several images per lesion), so leakage between A, B and the pool **cannot be ruled out**; report this as a limitation.

**Arms** (all start from the same base model; 2 epochs, max lr 5e-4, one-cycle, AdamW unless stated):
- `ft_full`: weights trained, BN statistics updating, BN scale/shift trainable (the Step 6 recipe).
- `ft_statsfrozen`: BN statistics frozen (eval mode), BN scale/shift trainable.
- `ft_affinefrozen`: BN statistics updating, BN scale/shift frozen.
- `ft_bnfrozen`: both frozen.
- `adabn`: no gradient step; BN running statistics re-estimated on the data in one pass (label-free).
- Intensity: `ft_full` and `ft_bnfrozen` at 0.5, 1, 2, 4 epochs on seen data.
- `fresh_half:ft_full`: fine-tuned on B1 only, calibrated on labeled B2 (**splitting baseline**, `split_fresh`).
Each arm runs on **seen** (A) and **fresh** (B) data except the splitting baseline.

**Methods.** `reuse` (base threshold), `sizematch` (label-free), `recal` (labeled pool calibration half), `split_fresh` (labeled B2).

**Hypotheses (declared before running; contrasts use APS reuse gap in points, paired within seed, 95% t-intervals over seeds, no p-values).**
- **H1 (primary):** BN statistics updating changes the coverage gap: `ft_full - ft_statsfrozen` differs from 0.
- **H2 (primary):** seen vs fresh data changes it: `fresh:ft_full - seen:ft_full` differs from 0.
- H3: `adabn` alone produces a nonzero gap.
- H4: `split_fresh` has smaller |gap| than `sizematch` when labeled fresh data exist (the case where `sizematch` is not needed).
- Secondary: BN scale/shift effect; intensity dependence; per-class effects.
A result that contradicts a hypothesis is reported as such. The seed count (10) is fixed in advance.

**Score-shift diagnostics (Experiment C), defined before running.** On the labeled pool, for each example i with true label y_i, let s_i be the base model's score and s'_i the adapted model's. Define the shift D_i = s'_i - s_i, eps_p = the p-th quantile of |D_i| (p = 0.5, 0.9, 0.99), and KS = the two-sample Kolmogorov-Smirnov distance between {s_i} and {s'_i}. With q0 the base threshold, the reuse gap equals F_s'(q0) - F_s(q0) **by definition**, so |gap| <= KS holds as an identity, not as an explanation. The explanatory content is the decomposition: change in accuracy, in the true-label probability, in the true-label rank, in mean max-probability and entropy, and in mean set size at q0 (the last three need no labels). We report Spearman correlations between these shifts and the observed gap across (seed, arm) cells. A small shift is **not** a coverage guarantee.

**Confounders to keep in mind.** Fewer base-training images (half the training set) than Steps 1-6; the one-cycle schedule restarts at fine-tuning; B1 is half the size of B; the splitting baseline's calibration set (B2) is similar in size to the pool's calibration half; DermaMNIST lesion duplicates.

Run all cells; check `smoke: False`. Results are saved per (dataset, architecture, seed) and finished seeds are skipped on re-run (`USE_DRIVE = True` protects against restarts). After the first seed the notebook prints elapsed time so you can estimate the total; I estimate about 2-3 hours for 10 seeds on a GPU, but I have not measured it.

In [ ]:
!pip install -q medmnist scipy
import os, sys, copy, math, time, glob, itertools
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F

# ---- CONFIG ----
SMOKE = False                      # True = synthetic data, tests the code only (never a result)
DATASETS = ["dermamnist", "bloodmnist"]   # drop names to run fewer
SEEDS = list(range(10))
EPOCHS = {"pathmnist": 8, "dermamnist": 15, "bloodmnist": 12}
FT_EPOCHS = 2                      # default fine-tune epochs
INTENSITY = [0.5, 1, 4]              # extra fine-tune lengths (epochs) for ft_full and ft_bnfrozen on seen data
ARCHS = ["resnet", "plain"]
N_SPLITS = 40                      # random calibration/eval splits per seed
ALPHA = 0.1
USE_DRIVE = False                  # True = save results to Drive so a runtime restart does not lose finished datasets
OUT_DIR = "/content/step7_out"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if USE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive"); OUT_DIR = "/content/drive/MyDrive/step7_out"
os.makedirs(OUT_DIR, exist_ok=True)
print("device:", DEVICE, "| smoke:", SMOKE, "| datasets:", DATASETS, "| seeds:", SEEDS, "| out:", OUT_DIR)
if DEVICE == "cpu": print("WARNING: CPU only. Switch to GPU (Runtime > Change runtime type) or run one dataset and one seed.")

def set_seed(s):
    np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

## Code

In [ ]:
# %%
def lac_all(probs):
    """Score of every label: 1 - p(label). probs: (n, K)."""
    return 1.0 - probs

def aps_all(probs):
    """Deterministic APS score of every label: total prob mass of labels ranked at or above it."""
    order = np.argsort(-probs, axis=1)
    cum = np.cumsum(np.take_along_axis(probs, order, axis=1), axis=1)
    ranks = np.argsort(order, axis=1)
    return np.take_along_axis(cum, ranks, axis=1)

def raps_all(probs, lam=0.05, k_reg=2):
    """Deterministic RAPS-style score: APS mass plus lam * max(0, rank - k_reg). Fixed hyperparameters, not tuned."""
    order = np.argsort(-probs, axis=1)
    cum = np.cumsum(np.take_along_axis(probs, order, axis=1), axis=1)
    ranks = np.argsort(order, axis=1)
    pen = lam * np.maximum(0, np.arange(1, probs.shape[1] + 1)[None, :] - k_reg)
    return np.take_along_axis(cum + pen, ranks, axis=1)

SCORES = {"LAC": lac_all, "APS": aps_all, "RAPS": raps_all}

def conformal_q(true_scores, alpha=ALPHA):
    """Finite-sample split-conformal quantile. Returns inf if the calibration set is too small."""
    n = len(true_scores)
    k = int(math.ceil((n + 1) * (1 - alpha)))
    if n == 0 or k > n:
        return np.inf
    return float(np.sort(true_scores)[k - 1])

def summarize(sets, y, min_class_n=10):
    covered = sets[np.arange(len(y)), y]
    cls = [covered[y == c].mean() for c in np.unique(y) if (y == c).sum() >= min_class_n]
    return dict(coverage=float(covered.mean()), size=float(sets.sum(1).mean()),
                worst_class_cov=float(min(cls)) if cls else np.nan)


In [ ]:
# %%
class Block(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.b1 = nn.BatchNorm2d(cout)
        self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False);     self.b2 = nn.BatchNorm2d(cout)
        self.sc = None
        if stride != 1 or cin != cout:
            self.sc = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
    def forward(self, x):
        o = F.relu(self.b1(self.c1(x))); o = self.b2(self.c2(o))
        return F.relu(o + (x if self.sc is None else self.sc(x)))

class ResNet18Small(nn.Module):
    """ResNet-18 layout (2,2,2,2 blocks) with a 3x3 stem, suited to 28x28 images."""
    def __init__(self, num_classes, width=64):
        super().__init__()
        w = width
        self.stem = nn.Sequential(nn.Conv2d(3, w, 3, 1, 1, bias=False), nn.BatchNorm2d(w), nn.ReLU())
        cfg, layers, cin = [(w, 1), (w, 1), (2*w, 2), (2*w, 1), (4*w, 2), (4*w, 1), (8*w, 2), (8*w, 1)], [], w
        for cout, s in cfg:
            layers.append(Block(cin, cout, s)); cin = cout
        self.layers = nn.Sequential(*layers); self.fc = nn.Linear(cin, num_classes)
    def forward(self, x):
        x = self.layers(self.stem(x))
        return self.fc(F.adaptive_avg_pool2d(x, 1).flatten(1))

def train_model(model, X, y, epochs, lr=2e-3, bs=256, seed=0):
    set_seed(seed); model.to(DEVICE).train()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * math.ceil(len(y) / bs))
    Xt, yt = torch.as_tensor(X), torch.as_tensor(y).long()
    for ep in range(epochs):
        perm = torch.randperm(len(yt))
        for i in range(0, len(yt), bs):
            idx = perm[i:i + bs]
            xb, yb = Xt[idx].to(DEVICE), yt[idx].to(DEVICE)
            loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return model.eval()

@torch.no_grad()
def predict_probs(model, X, bs=512):
    model.to(DEVICE).eval(); out = []
    Xt = torch.as_tensor(X)
    for i in range(0, len(Xt), bs):
        out.append(F.softmax(model(Xt[i:i + bs].to(DEVICE)), dim=1).cpu())
    return torch.cat(out).numpy().astype(np.float64)

class PlainCNN(nn.Module):
    """VGG-style CNN, no skip connections: conv-BN-ReLU x6 with two max-pools, then global average pool and a linear layer."""
    def __init__(self, num_classes, width=64):
        super().__init__(); w = width
        cb = lambda i, o: [nn.Conv2d(i, o, 3, 1, 1, bias=False), nn.BatchNorm2d(o), nn.ReLU()]
        layers = cb(3, w) + cb(w, w) + [nn.MaxPool2d(2)] + cb(w, 2*w) + cb(2*w, 2*w) + [nn.MaxPool2d(2)] + cb(2*w, 4*w) + cb(4*w, 4*w)
        self.body = nn.Sequential(*layers); self.fc = nn.Linear(4*w, num_classes)
    def forward(self, x):
        return self.fc(F.adaptive_avg_pool2d(self.body(x), 1).flatten(1))

ARCH = "resnet"
def build_model(arch, K):
    w = 16 if SMOKE else 64
    return ResNet18Small(K, width=w) if arch == "resnet" else PlainCNN(K, width=w)


def bn_modules(m):
    return [x for x in m.modules() if isinstance(x, nn.modules.batchnorm._BatchNorm)]

def adapt(model, X, y, epochs, bn_stats="update", bn_affine="train", lr=5e-4, bs=256, seed=0):
    """Fine-tune conv/linear weights. bn_stats: 'update' (BN in train mode) or 'frozen' (BN in eval mode).
    bn_affine: 'train' or 'frozen' (BN scale and shift). One-cycle schedule over round(epochs * steps_per_epoch) steps."""
    m = copy.deepcopy(model); set_seed(seed + 99); m.to(DEVICE).train()
    bns = bn_modules(m)
    if bn_stats == "frozen":
        for b in bns: b.eval()
    if bn_affine == "frozen":
        for b in bns:
            for p in b.parameters(): p.requires_grad_(False)
    params = [p for p in m.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    total = max(1, int(round(epochs * math.ceil(len(y) / bs))))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=total)
    Xt, yt = torch.as_tensor(X), torch.as_tensor(y).long(); steps = 0
    while steps < total:
        perm = torch.randperm(len(yt))
        for i in range(0, len(yt), bs):
            if steps >= total: break
            idx = perm[i:i + bs]
            loss = F.cross_entropy(m(Xt[idx].to(DEVICE)), yt[idx].to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step(); sched.step(); steps += 1
    for p in m.parameters(): p.requires_grad_(True)
    return m.eval()

@torch.no_grad()
def adabn(model, X, bs=256):
    """Label-free: re-estimate BN running statistics on X (cumulative average over one pass); no weight update."""
    m = copy.deepcopy(model).to(DEVICE).train()
    for b in bn_modules(m): b.reset_running_stats(); b.momentum = None
    Xt = torch.as_tensor(X)
    for i in range(0, len(Xt), bs): m(Xt[i:i + bs].to(DEVICE))
    for b in bn_modules(m): b.momentum = 0.1
    return m.eval()

def strat_split(y, frac, rng):
    a, b = [], []
    for c in np.unique(y):
        idx = rng.permutation(np.where(y == c)[0]); k = int(round(len(idx) * frac))
        if len(idx) >= 2: k = min(max(k, 1), len(idx) - 1)
        a.append(idx[:k]); b.append(idx[k:])
    return np.concatenate(a), np.concatenate(b)

def conformal_q(true_scores, alpha=ALPHA):
    n = len(true_scores); k = int(math.ceil((n + 1) * (1 - alpha)))
    if n == 0 or k > n: return np.inf
    return float(np.sort(true_scores)[k - 1])

def summarize(sets, y, min_class_n=5):
    covered = sets[np.arange(len(y)), y]
    cls = [covered[y == c].mean() for c in np.unique(y) if (y == c).sum() >= min_class_n]
    return dict(coverage=float(covered.mean()), size=float(sets.sum(1).mean()),
                worst_class_cov=float(min(cls)) if cls else np.nan,
                cls_dev=float(np.mean(np.abs(np.array(cls) - (1 - ALPHA)))) if cls else np.nan)

def mean_size(S, q):
    return float((S <= q).sum(1).mean())

def size_match_q(S_c, target_size, iters=40):
    """Label-free: smallest threshold whose mean set size on unlabeled inputs reaches target_size (bisection)."""
    lo, hi = 0.0, float(S_c.max()) + 1e-9
    for _ in range(iters):
        mid = (lo + hi) / 2
        if mean_size(S_c, mid) < target_size: lo = mid
        else: hi = mid
    return hi

def make_arms(base, XA, yA, XB, yB, XB1, yB1, seed):
    """name -> (model, group). Models are built lazily in order; all start from the same base model."""
    arms = {}
    for d, (X, y) in {"seen": (XA, yA), "fresh": (XB, yB)}.items():
        arms[f"{d}:ft_full@2"]        = lambda X=X, y=y: adapt(base, X, y, FT_EPOCHS, "update", "train", seed=seed)
        arms[f"{d}:ft_statsfrozen@2"] = lambda X=X, y=y: adapt(base, X, y, FT_EPOCHS, "frozen", "train", seed=seed)
        arms[f"{d}:ft_affinefrozen@2"] = lambda X=X, y=y: adapt(base, X, y, FT_EPOCHS, "update", "frozen", seed=seed)
        arms[f"{d}:ft_bnfrozen@2"]    = lambda X=X, y=y: adapt(base, X, y, FT_EPOCHS, "frozen", "frozen", seed=seed)
        arms[f"{d}:adabn"]            = lambda X=X: adabn(base, X)
    for e in INTENSITY:
        arms[f"seen:ft_full@{e}"]     = lambda e=e: adapt(base, XA, yA, e, "update", "train", seed=seed)
        arms[f"seen:ft_bnfrozen@{e}"] = lambda e=e: adapt(base, XA, yA, e, "frozen", "frozen", seed=seed)
    arms["fresh_half:ft_full@2"]      = lambda: adapt(base, XB1, yB1, FT_EPOCHS, "update", "train", seed=seed)
    return arms

def shift_stats(P0, P1, y, q0):
    """Score-shift diagnostics on the labeled pool (APS). P0 base probs, P1 adapted probs, q0 base threshold."""
    from scipy.stats import ks_2samp
    S0, S1 = aps_all(P0), aps_all(P1); n = len(y); ar = np.arange(n)
    s0, s1 = S0[ar, y], S1[ar, y]; D = np.abs(s1 - s0)
    ent = lambda P: float(np.mean(-(P * np.log(P + 1e-12)).sum(1)))
    r0 = (P0 > P0[ar, y][:, None]).sum(1); r1 = (P1 > P1[ar, y][:, None]).sum(1)
    return dict(eps50=float(np.quantile(D, .5)), eps90=float(np.quantile(D, .9)), eps99=float(np.quantile(D, .99)),
                mean_shift=float(np.mean(s1 - s0)), ks=float(ks_2samp(s0, s1).statistic),
                cdf_id=float(np.mean(s1 <= q0) - np.mean(s0 <= q0)),
                d_acc=float(np.mean(P1.argmax(1) == y) - np.mean(P0.argmax(1) == y)),
                d_ptrue=float(np.mean(P1[ar, y]) - np.mean(P0[ar, y])),
                d_maxp=float(P1.max(1).mean() - P0.max(1).mean()), d_ent=ent(P1) - ent(P0),
                d_size=mean_size(S1, q0) - mean_size(S0, q0),
                frac_rank_changed=float(np.mean(r0 != r1)), frac_top1_changed=float(np.mean(P0.argmax(1) != P1.argmax(1))))


## Data and experiment

In [ ]:
def load_medmnist(name, smoke):
    if smoke:
        rng = np.random.RandomState(0); K = 7
        def make(n):
            y = rng.randint(0, K, n); X = rng.rand(n, 3, 28, 28).astype(np.float32) * 0.8
            for k in range(K):
                r, c, ch = (k // 3) * 9 + 2, (k % 3) * 9 + 2, k % 3
                X[y == k, ch, r:r + 8, c:c + 8] += 0.6
            return (X - 0.5) / 0.5, y
        Xtr, ytr = make(2500); Xev, yev = make(1500)
        return Xtr, ytr, Xev, yev, K
    import medmnist
    from medmnist import INFO
    cls = getattr(medmnist, INFO[name]["python_class"])
    f = lambda a: ((a.imgs.astype(np.float32) / 255.0 - 0.5) / 0.5).transpose(0, 3, 1, 2).copy()
    lab = lambda a: a.labels.squeeze().astype(np.int64)
    tr, va, te = (cls(split=s, download=True) for s in ("train", "val", "test"))
    K = len(INFO[name]["label"])
    Xev, yev = np.concatenate([f(va), f(te)]), np.concatenate([lab(va), lab(te)])
    return f(tr), lab(tr), Xev, yev, K

def run_seed(name, arch, seed, Xtr, ytr, Xev, yev, K):
    global ARCH
    ARCH = arch; t0 = time.time(); epochs = 3 if SMOKE else EPOCHS[name]
    rng = np.random.RandomState(777 + seed)
    iA, iB = strat_split(ytr, 0.5, rng); iB1, iB2 = strat_split(ytr[iB], 0.5, rng); iB1, iB2 = iB[iB1], iB[iB2]
    XA, yA, XB, yB, XB1, yB1, XB2, yB2 = Xtr[iA], ytr[iA], Xtr[iB], ytr[iB], Xtr[iB1], ytr[iB1], Xtr[iB2], ytr[iB2]
    base = train_model(build_model(arch, K), XA, yA, epochs, seed=seed)
    P0 = predict_probs(base, Xev); probs = {"base": P0}; extra = {}
    for aname, mk in make_arms(base, XA, yA, XB, yB, XB1, yB1, seed).items():
        m = mk(); probs[aname] = predict_probs(m, Xev)
        if aname == "fresh_half:ft_full@2": extra["B2"] = predict_probs(m, XB2)
    accs = {a: float(np.mean(p.argmax(1) == yev)) for a, p in probs.items()}
    rows, srows = [], []
    for sname, sfn in SCORES.items():
        S = {a: sfn(p) for a, p in probs.items()}
        S_B2 = sfn(extra["B2"])[np.arange(len(yB2)), yB2]
        for sp in range(N_SPLITS):
            r_ = np.random.RandomState(1000 * seed + sp)
            perm = r_.permutation(len(yev)); cal, ev = perm[:len(yev) // 2], perm[len(yev) // 2:]
            q0 = conformal_q(S["base"][cal, yev[cal]]); target = mean_size(S["base"][cal], q0)
            for a in probs:
                qs = [("reuse", q0), ("sizematch", size_match_q(S[a][cal], target)), ("recal", conformal_q(S[a][cal, yev[cal]]))]
                if a == "fresh_half:ft_full@2": qs.append(("split_fresh", conformal_q(S_B2)))
                for mode, q in qs:
                    rows.append(dict(dataset=name, arch=arch, seed=seed, split=sp, score=sname, arm=a, mode=mode, acc=accs[a],
                                     n_cal=(len(yB2) if mode == "split_fresh" else len(cal)), **summarize(S[a][ev] <= q, yev[ev])))
    q_pool = conformal_q(aps_all(P0)[np.arange(len(yev)), yev])
    for a in probs:
        if a != "base": srows.append(dict(dataset=name, arch=arch, seed=seed, arm=a, **shift_stats(P0, probs[a], yev, q_pool)))
    print(f"  [{name}|{arch}] seed {seed} done in {time.time()-t0:.0f}s | base acc {accs['base']:.3f} | " +
          " ".join(f"{a.split(':')[1] if ':' in a else a}={accs[a]:.3f}" for a in list(probs)[1:4]), flush=True)
    return pd.DataFrame(rows), pd.DataFrame(srows)

def run_cell(name, arch):
    Xtr, ytr, Xev, yev, K = load_medmnist(name, SMOKE)
    print(f"[{name} | {arch}] train {len(ytr)} -> A {len(ytr)//2} / B {len(ytr)-len(ytr)//2} | pool {len(yev)} | classes {K}")
    R, Sx = [], []
    for seed in SEEDS:
        f1 = os.path.join(OUT_DIR, f"{name}_{arch}_s{seed}.csv"); f2 = os.path.join(OUT_DIR, f"{name}_{arch}_s{seed}_shift.csv")
        if os.path.exists(f1) and os.path.exists(f2) and not SMOKE:
            print(f"  seed {seed}: found saved results, loading"); r, s = pd.read_csv(f1), pd.read_csv(f2)
        else:
            r, s = run_seed(name, arch, seed, Xtr, ytr, Xev, yev, K)
            if not SMOKE: r.to_csv(f1, index=False); s.to_csv(f2, index=False)
        R.append(r); Sx.append(s)
    return pd.concat(R), pd.concat(Sx)


## Reporting

In [ ]:
from scipy import stats
PRINT_SCORES = ["APS"]          # LAC and RAPS are saved in the CSV files; set to ["APS","LAC","RAPS"] to print all
def ci95(x):
    x = np.asarray(x, float); x = x[~np.isnan(x)]; n = len(x)
    return float("nan") if n < 2 else stats.t.ppf(0.975, n - 1) * x.std(ddof=1) / np.sqrt(n)
def pm(x): return "{:+.2f} +/- {:.2f}".format(np.nanmean(x), ci95(x))

def per_seed(df):
    p = df.groupby(["seed", "score", "arm", "mode"])[["coverage", "size", "worst_class_cov", "cls_dev", "acc", "n_cal"]].mean().reset_index()
    p["gap"] = 100 * (p["coverage"] - (1 - ALPHA))
    rec = p[p["mode"] == "recal"].set_index(["seed", "score", "arm"])["size"]
    p["size_x"] = [r["size"] / rec[(r["seed"], r["score"], r["arm"])] for _, r in p.iterrows()]
    return p

def g(p, score, arm, mode): return p[(p["score"] == score) & (p["arm"] == arm) & (p["mode"] == mode)].set_index("seed")

def contrast(p, score, a1, m1, a2, m2, col="gap"):
    d = g(p, score, a1, m1)[col] - g(p, score, a2, m2)[col]
    return d

def report_cell(df, shift):
    if SMOKE: print("*** SMOKE = True: SYNTHETIC DATA, NOT A RESULT ***")
    name, arch = df["dataset"].iloc[0], df["arch"].iloc[0]; tag = f"{name} | {arch}"
    p = per_seed(df); ns = p["seed"].nunique()
    print(f"\n################ {tag}  ({ns} seeds) ################")
    acc = p[(p["mode"] == "reuse") & (p["score"] == "APS")].groupby("arm")["acc"].mean().round(4)
    print("\nTop-1 accuracy on the pool (mean over seeds):"); print(acc.to_string())
    for sc in PRINT_SCORES:
        print(f"\n===== {tag} | {sc}: coverage gap in points (negative = undercoverage), mean +/- 95% t over {ns} seeds =====")
        t = []
        for a in [x for x in p["arm"].unique() if x != "base"]:
            row = dict(arm=a)
            for m in ["reuse", "sizematch", "recal", "split_fresh"]:
                gg = g(p, sc, a, m)
                row[m] = pm(gg["gap"]) if len(gg) else "-"
            t.append(row)
        print(pd.DataFrame(t).to_string(index=False))
        print(f"\n----- {tag} | {sc}: worst-class coverage and mean set size relative to recal (reuse / sizematch / recal) -----")
        t = []
        for a in [x for x in p["arm"].unique() if x != "base"]:
            row = dict(arm=a)
            for m in ["reuse", "sizematch", "recal"]:
                gg = g(p, sc, a, m); row[f"wc_{m}"] = round(gg["worst_class_cov"].mean(), 3)
            for m in ["reuse", "sizematch"]:
                row[f"size_x_{m}"] = round(g(p, sc, a, m)["size_x"].mean(), 2)
            t.append(row)
        print(pd.DataFrame(t).to_string(index=False))
    sc = "APS"
    print(f"\n===== {tag} | PRIMARY AND SECONDARY CONTRASTS ({sc}, reuse gap in points, paired within seed) =====")
    C = [("H1 BN statistics: ft_full - ft_statsfrozen (seen)", contrast(p, sc, "seen:ft_full@2", "reuse", "seen:ft_statsfrozen@2", "reuse")),
         ("H2 seen vs fresh: fresh:ft_full - seen:ft_full", contrast(p, sc, "fresh:ft_full@2", "reuse", "seen:ft_full@2", "reuse")),
         ("H3 adabn only (seen): gap itself", g(p, sc, "seen:adabn", "reuse")["gap"]),
         ("   adabn only (fresh): gap itself", g(p, sc, "fresh:adabn", "reuse")["gap"]),
         ("S1 BN stats, affine frozen: ft_affinefrozen - ft_bnfrozen (seen)", contrast(p, sc, "seen:ft_affinefrozen@2", "reuse", "seen:ft_bnfrozen@2", "reuse")),
         ("S2 BN affine, stats updating: ft_full - ft_affinefrozen (seen)", contrast(p, sc, "seen:ft_full@2", "reuse", "seen:ft_affinefrozen@2", "reuse")),
         ("S3 everything BN frozen vs full: ft_bnfrozen - ft_full (seen)", contrast(p, sc, "seen:ft_bnfrozen@2", "reuse", "seen:ft_full@2", "reuse"))]
    print(pd.DataFrame([dict(contrast=n, mean_ci=pm(d)) for n, d in C]).to_string(index=False))
    print("(Gap is signed: a negative contrast means the first arm undercovers more.)")
    print(f"\n===== {tag} | H4: splitting baseline vs sizematch vs reuse on the fresh_half arm ({sc}) =====")
    a = "fresh_half:ft_full@2"; rows = []
    for m in ["reuse", "sizematch", "split_fresh", "recal"]:
        gg = g(p, sc, a, m); rows.append(dict(method=m, gap=pm(gg["gap"]), abs_gap=round(gg["gap"].abs().mean(), 2), size_x=round(gg["size_x"].mean(), 2), wc=round(gg["worst_class_cov"].mean(), 3), n_labels=int(gg["n_cal"].mean()) if m in ("split_fresh", "recal") else 0))
    print(pd.DataFrame(rows).to_string(index=False))
    print("(sizematch uses 0 labels; split_fresh uses the n_labels fresh labels held out from fine-tuning; recal uses the pool calibration half.)")
    print(f"\n===== {tag} | INTENSITY ({sc}, gap in points) =====")
    rows = []
    for fam in ["ft_full", "ft_bnfrozen"]:
        for e in [0.5, 1, 2, 4]:
            a = f"seen:{fam}@{e}"
            if a in p["arm"].values:
                rows.append(dict(arm=fam, epochs=e, reuse=pm(g(p, sc, a, "reuse")["gap"]), sizematch=pm(g(p, sc, a, "sizematch")["gap"])))
    print(pd.DataFrame(rows).to_string(index=False))
    print(f"\n===== {tag} | SCORE-SHIFT DIAGNOSTICS (APS, mean over seeds) =====")
    sh = shift.groupby("arm")[["eps50", "eps90", "eps99", "ks", "mean_shift", "d_acc", "d_ptrue", "d_maxp", "d_ent", "d_size", "frac_rank_changed", "frac_top1_changed"]].mean().round(4)
    print(sh.to_string())
    return p

def summary_all(ps, shifts):
    print("\n################ ACROSS CELLS: primary contrasts (mean over seeds in each cell, APS reuse gap in points) ################")
    rows = []
    for k, p in ps.items():
        sc = "APS"
        rows.append(dict(cell=k,
            H1=pm(contrast(p, sc, "seen:ft_full@2", "reuse", "seen:ft_statsfrozen@2", "reuse")),
            H2=pm(contrast(p, sc, "fresh:ft_full@2", "reuse", "seen:ft_full@2", "reuse")),
            adabn_seen=pm(g(p, sc, "seen:adabn", "reuse")["gap"]),
            full_seen=pm(g(p, sc, "seen:ft_full@2", "reuse")["gap"]),
            bnfrozen_seen=pm(g(p, sc, "seen:ft_bnfrozen@2", "reuse")["gap"])))
    print(pd.DataFrame(rows).to_string(index=False))
    allp = pd.concat([p.assign(cell=k) for k, p in ps.items()])
    gp = allp[(allp["score"] == "APS") & (allp["mode"] == "reuse")][["cell", "seed", "arm", "gap"]]
    sh = pd.concat(shifts.values()).assign(cell=lambda d: d["dataset"] + "|" + d["arch"])
    j = sh.merge(gp, on=["cell", "seed", "arm"])
    print("\n################ SPEARMAN correlation of shift diagnostics with the APS reuse gap across (cell, seed, arm), n =", len(j), "################")
    out = []
    for c in ["eps50", "eps90", "eps99", "ks", "mean_shift", "d_acc", "d_ptrue", "d_maxp", "d_ent", "d_size", "frac_rank_changed", "frac_top1_changed"]:
        r, pv = stats.spearmanr(j[c], j["gap"]); out.append(dict(stat=c, spearman_rho=round(r, 3), label_free=c in ("d_maxp", "d_ent", "d_size", "frac_top1_changed")))
    print(pd.DataFrame(out).to_string(index=False))
    print("\nCaution: arms within a seed share a base model, so cells are not independent; read rho as descriptive, not as a test.")
    print("Read: (1) H1: does BN-stat updating move the gap, and in which direction, in every cell? (2) H2: does fresh data shrink it? (3) does adabn alone reproduce part of it? (4) H4: when labeled fresh data exist, is split_fresh better than sizematch? (5) which shift statistics track the gap, and are the label-free ones enough?")


## Run

In [ ]:
ps, shifts = {}, {}
for arch in ARCHS:
    for name in DATASETS:
        df, sh = run_cell(name, arch)
        ps[f"{name}|{arch}"] = report_cell(df, sh); shifts[f"{name}|{arch}"] = sh
summary_all(ps, shifts)